In [1]:
# Company Entity Resolution

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, lit, count, round, monotonically_increasing_id
from pyspark.storagelevel import StorageLevel

In [2]:
# Spark session
spark = (
    SparkSession.builder
    .appName("company-dedup")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

spark.sparkContext.setCheckpointDir("/tmp/spark_checkpoints")
spark.conf.set("spark.sql.shuffle.partitions", "200")

In [3]:
# Reading the parquet input
df_raw = spark.read.parquet("veridion.parquet").cache()
print(df_raw.count())
df_raw.printSchema()

df_raw = df_raw.withColumn("record_id", monotonically_increasing_id())

df = df_raw

33446
root
 |-- company_name: string (nullable = true)
 |-- company_legal_names: string (nullable = true)
 |-- company_commercial_names: string (nullable = true)
 |-- main_country_code: string (nullable = true)
 |-- main_country: string (nullable = true)
 |-- main_region: string (nullable = true)
 |-- main_city_district: string (nullable = true)
 |-- main_city: string (nullable = true)
 |-- main_postcode: string (nullable = true)
 |-- main_street: string (nullable = true)
 |-- main_street_number: string (nullable = true)
 |-- main_latitude: string (nullable = true)
 |-- main_longitude: string (nullable = true)
 |-- main_address_raw_text: string (nullable = true)
 |-- locations: string (nullable = true)
 |-- num_locations: string (nullable = true)
 |-- company_type: string (nullable = true)
 |-- year_founded: string (nullable = true)
 |-- lnk_year_founded: string (nullable = true)
 |-- short_description: string (nullable = true)
 |-- long_description: string (nullable = true)
 |-- busin

In [4]:
# Completeness report 
n = df.count()
exprs = [
    round(count(when(col(c).isNotNull(), 1)) * 100 / lit(n), 2).alias(c)
    for c in df.columns
]
wide = df.select(*exprs)
pretty = wide.selectExpr(
    "stack({}, {}) as (column, completeness)".format(
        len(df.columns),
        ", ".join([f"'{c}', {c}" for c in df.columns])
    )
)
pretty.orderBy(col("completeness").desc()).show(200, truncate=False)

+----------------------------+------------+
|column                      |completeness|
+----------------------------+------------+
|status                      |100.0       |
|record_id                   |100.0       |
|created_at                  |99.87       |
|last_updated_at             |99.87       |
|company_name                |97.52       |
|website_url                 |95.36       |
|website_domain              |95.36       |
|website_tld                 |95.36       |
|main_country_code           |93.93       |
|main_country                |93.93       |
|locations                   |93.93       |
|main_region                 |90.03       |
|main_city                   |88.51       |
|company_commercial_names    |84.08       |
|main_address_raw_text       |83.66       |
|main_postcode               |71.22       |
|primary_phone               |68.17       |
|phone_numbers               |68.17       |
|main_street                 |59.74       |
|business_model              |59

In [5]:
# Normalization step 

# Normalization of domain/website
df = df.withColumn("website_url_norm", F.lower(F.trim(F.col("website_url"))))

df = df.withColumn("domain_from_url", F.regexp_replace(F.col("website_url_norm"), r"^https?://", ""))
df = df.withColumn("domain_from_url", F.regexp_replace(F.col("domain_from_url"), r"^www\.", ""))
df = df.withColumn("domain_from_url", F.regexp_replace(F.col("domain_from_url"), r"/.*$", ""))
df = df.withColumn("domain_from_url", F.regexp_replace(F.col("domain_from_url"), r"[\?#].*$", ""))
df = df.withColumn("domain_from_url", F.trim(F.col("domain_from_url")))

df = df.withColumn("website_domain_norm", F.lower(F.trim(F.col("website_domain"))))
df = df.withColumn("website_domain_norm", F.regexp_replace(F.col("website_domain_norm"), r"^www\.", ""))

df = df.withColumn(
    "domain_final",
    F.when(
        F.col("website_domain_norm").isNotNull() & (F.col("website_domain_norm") != ""),
        F.col("website_domain_norm")
    ).otherwise(F.col("domain_from_url"))
)

# nulling out the malformed domains
df = df.withColumn(
    "domain_final",
    F.when(F.col("domain_final").rlike(r"^[a-z0-9.-]+\.[a-z]{2,}$"), F.col("domain_final"))
     .otherwise(F.lit(None))
)

df.select(
    "website_domain",
    "website_url",
    "domain_from_url",
    "website_domain_norm",
    "domain_final"
).show(20, truncate=False)

# handling of TLD
df = df.withColumn("website_tld_norm", F.lower(F.trim(F.col("website_tld"))))
df = df.withColumn(
    "tld_from_domain",
    F.when(
        F.col("domain_final").isNotNull(),
        F.regexp_extract(F.col("domain_final"), r"\.([a-z]{2,})$", 1)
    ).otherwise(F.lit(None))
)

df = df.withColumn(
    "tld_final",
    F.when(
        F.col("website_tld_norm").isNotNull() & (F.col("website_tld_norm") != ""),
        F.regexp_replace(F.col("website_tld_norm"), r"^\.", "")
    ).otherwise(F.col("tld_from_domain"))
)

+------------------------+------------------------------------------------------------------------------------------------------------------------+------------------------------+------------------------+------------------------+
|website_domain          |website_url                                                                                                             |domain_from_url               |website_domain_norm     |domain_final            |
+------------------------+------------------------------------------------------------------------------------------------------------------------+------------------------------+------------------------+------------------------+
|owensliquors.com        |https://pawleysisland.owensliquors.com/                                                                                 |pawleysisland.owensliquors.com|owensliquors.com        |owensliquors.com        |
|clubtarneit.com.au      |https://www.clubtarneit.com.au/                           

In [6]:
# Normalization of company names
LEGAL_SUFFIX_RE = r"\b(incorporated|inc|llc|l\.l\.c|ltd|limited|gmbh|srl|s\.r\.l|sa|s\.a|spa|s\.p\.a|bv|b\.v|ag|plc|kg|oy|ab|as|pte|co|company|corp|corporation)\b"

def normalize_name(colname: str, outcol: str):
    return (
        F.trim(
            F.regexp_replace(
                F.regexp_replace(
                    F.regexp_replace(F.lower(F.trim(F.col(colname))), r"[^a-z0-9\s]", " "),
                    LEGAL_SUFFIX_RE, " "
                ),
                r"\s+", " "
            )
        ).alias(outcol)
    )

df = df.withColumn("company_name_norm", normalize_name("company_name", "company_name_norm"))
df = df.withColumn("company_commercial_names_norm", normalize_name("company_commercial_names", "company_commercial_names_norm"))
df = df.withColumn("company_legal_names_norm", normalize_name("company_legal_names", "company_legal_names_norm"))


In [7]:
# Normalization of phones
df = df.withColumn("primary_phone_norm", F.trim(F.col("primary_phone")))
df = df.withColumn("primary_phone_norm", F.regexp_replace(F.col("primary_phone_norm"), r"[^\d+]", ""))
df = df.withColumn("primary_phone_norm", F.when(F.col("primary_phone_norm") == "", F.lit(None)).otherwise(F.col("primary_phone_norm")))

df = df.withColumn("phone_numbers_raw", F.trim(F.col("phone_numbers")))
df = df.withColumn(
    "phone_numbers_arr",
    F.when(F.col("phone_numbers_raw").isNotNull(),
           F.split(F.col("phone_numbers_raw"), r"\s*\|\s*|\s*,\s*|\s*;\s*")
          ).otherwise(F.array())
)
df = df.withColumn(
    "phone_numbers_norm_arr",
    F.expr("transform(phone_numbers_arr, x -> regexp_replace(trim(x), '[^0-9+]', ''))")
)
df = df.withColumn(
    "phone_numbers_norm_arr",
    F.expr("filter(phone_numbers_norm_arr, x -> x is not null and x <> '')")
)
df = df.withColumn(
    "phone_any_norm",
    F.when(F.size(F.col("phone_numbers_norm_arr")) > 0, F.col("phone_numbers_norm_arr")[0])
     .otherwise(F.lit(None))
)


In [8]:
# Normalization of emails
df = df.withColumn("primary_email_norm", F.lower(F.trim(F.col("primary_email"))))
df = df.withColumn(
    "primary_email_norm",
    F.when(F.col("primary_email_norm").rlike(r"^[^@\s]+@[^@\s]+\.[^@\s]+$"), F.col("primary_email_norm"))
     .otherwise(F.lit(None))
)
df = df.withColumn(
    "email_domain",
    F.when(F.col("primary_email_norm").isNotNull(),
           F.regexp_extract(F.col("primary_email_norm"), r"@([^@]+)$", 1)
          ).otherwise(F.lit(None))
)

In [9]:
# Normalization of geography/address
df = df.withColumn("country_code_norm", F.upper(F.trim(F.col("main_country_code"))))

df = df.withColumn("main_city_norm", F.lower(F.trim(F.col("main_city"))))
df = df.withColumn("main_city_norm", F.regexp_replace(F.col("main_city_norm"), r"\s+", " "))

df = df.withColumn("main_postcode_norm", F.upper(F.trim(F.col("main_postcode"))))
df = df.withColumn("main_postcode_norm", F.regexp_replace(F.col("main_postcode_norm"), r"\s+", ""))
df = df.withColumn("main_postcode_norm", F.when(F.col("main_postcode_norm") == "", F.lit(None)).otherwise(F.col("main_postcode_norm")))

df = df.withColumn("main_address_raw_norm", F.lower(F.trim(F.col("main_address_raw_text"))))
df = df.withColumn("main_address_raw_norm", F.regexp_replace(F.col("main_address_raw_norm"), r"\s+", " "))

df = df.withColumn("main_street_norm", F.lower(F.trim(F.col("main_street"))))
df = df.withColumn("main_street_norm", F.regexp_replace(F.col("main_street_norm"), r"\s+", " "))

df = df.withColumn("main_street_number_norm", F.upper(F.trim(F.col("main_street_number"))))
df = df.withColumn("main_street_number_norm", F.when(F.col("main_street_number_norm") == "", F.lit(None)).otherwise(F.col("main_street_number_norm")))

df = df.withColumn("main_latitude_d", F.col("main_latitude").cast("double"))
df = df.withColumn("main_longitude_d", F.col("main_longitude").cast("double"))
df = df.withColumn("lat_r3", F.round(F.col("main_latitude_d"), 3))
df = df.withColumn("lon_r3", F.round(F.col("main_longitude_d"), 3))

In [10]:
# Blocking keys

# Domain block
df = df.withColumn(
    "block_domain",
    F.when(F.col("domain_final").isNotNull() & (F.col("domain_final") != ""),
           F.concat(F.lit("dom:"), F.col("domain_final")))
)

# Phones blocks
df = df.withColumn(
    "block_phones",
    F.when(F.col("phone_numbers_norm_arr").isNotNull(),
           F.expr("transform(phone_numbers_norm_arr, x -> concat('ph:', x))"))
     .otherwise(F.array())
)

df = df.withColumn(
    "block_phone_any",
    F.when(F.col("phone_any_norm").isNotNull() & (F.col("phone_any_norm") != ""),
           F.concat(F.lit("ph:"), F.col("phone_any_norm")))
)

# Email block
df = df.withColumn(
    "block_email",
    F.when(F.col("primary_email_norm").isNotNull() & (F.col("primary_email_norm") != ""),
           F.concat(F.lit("em:"), F.col("primary_email_norm")))
)

# Name head (from company_name_norm for blocking only)
df = df.withColumn(
    "name_head_block",
    F.when(F.col("company_name_norm").isNotNull() & (F.col("company_name_norm") != ""),
           F.element_at(F.split(F.col("company_name_norm"), " "), 1))
)

df = df.withColumn(
    "block_email_domain_name",
    F.when(
        F.col("email_domain").isNotNull() & (F.col("email_domain") != "") &
        F.col("name_head_block").isNotNull() & (F.col("name_head_block") != ""),
        F.concat(F.lit("emd:"), F.col("email_domain"), F.lit("|"), F.col("name_head_block"))
    )
)

# Geo blocks
df = df.withColumn(
    "block_country_city",
    F.when(
        F.col("country_code_norm").isNotNull() & (F.col("country_code_norm") != "") &
        F.col("main_city_norm").isNotNull() & (F.col("main_city_norm") != ""),
        F.concat(F.lit("geo1:"), F.col("country_code_norm"), F.lit("|"), F.col("main_city_norm"))
    )
)

df = df.withColumn(
    "block_country_postcode",
    F.when(
        F.col("country_code_norm").isNotNull() & (F.col("country_code_norm") != "") &
        F.col("main_postcode_norm").isNotNull() & (F.col("main_postcode_norm") != ""),
        F.concat(F.lit("geo2:"), F.col("country_code_norm"), F.lit("|"), F.col("main_postcode_norm"))
    )
)

df = df.withColumn(
    "block_geocoord",
    F.when(
        F.col("lat_r3").isNotNull() & F.col("lon_r3").isNotNull(),
        F.concat(F.lit("geo3:"), F.col("lat_r3").cast("string"), F.lit("|"), F.col("lon_r3").cast("string"))
    )
)

# Name blocks
df = df.withColumn("name_tokens", F.split(F.col("company_name_norm"), " "))

df = df.withColumn(
    "name_fp",
    F.when(F.size(F.col("name_tokens")) >= 2,
           F.concat_ws(" ", F.col("name_tokens")[0], F.col("name_tokens")[1]))
     .when(F.size(F.col("name_tokens")) == 1, F.col("name_tokens")[0])
)

df = df.withColumn(
    "block_name_country",
    F.when(
        F.col("name_fp").isNotNull() & (F.col("name_fp") != "") &
        F.col("country_code_norm").isNotNull() & (F.col("country_code_norm") != ""),
        F.concat(F.lit("nm1:"), F.col("country_code_norm"), F.lit("|"), F.col("name_fp"))
    )
)

df = df.withColumn(
    "block_name_country_postcode",
    F.when(
        F.col("name_fp").isNotNull() & (F.col("name_fp") != "") &
        F.col("country_code_norm").isNotNull() & (F.col("country_code_norm") != "") &
        F.col("main_postcode_norm").isNotNull() & (F.col("main_postcode_norm") != ""),
        F.concat(F.lit("nm2:"), F.col("country_code_norm"), F.lit("|"), F.col("name_fp"), F.lit("|"), F.col("main_postcode_norm"))
    )
)

df = df.withColumn(
    "blocking_keys",
    F.array(
        "block_domain",
        "block_phone_any",
        "block_email",
        "block_email_domain_name",
        "block_country_city",
        "block_country_postcode",
        "block_geocoord",
        "block_name_country",
        "block_name_country_postcode"
    )
)

df = df.withColumn("blocking_keys", F.expr("filter(blocking_keys, x -> x is not null)"))
df = df.withColumn("blocking_keys", F.array_distinct(F.concat(F.col("blocking_keys"), F.col("block_phones"))))

df.select("record_id", "company_name", "domain_final", "blocking_keys").show(20, truncate=False)

blocks = df.select("record_id", F.explode("blocking_keys").alias("block_key"))
block_sizes = blocks.groupBy("block_key").count().orderBy(F.desc("count"))
block_sizes.show(50, truncate=False)

+-----------+------------------------------------------------+------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|record_id  |company_name                                    |domain_final            |blocking_keys                                                                                                                                                                                                                                                                            |
+-----------+------------------------------------------------+------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [11]:
# Matching (Deterministic+Fuzzy) with Anti Over-Merge

df = df.withColumn(
    "best_name_norm",
    F.coalesce(
        F.col("company_name_norm"),
        F.col("company_commercial_names_norm"),
        F.col("company_legal_names_norm")
    )
)

# Deterministic edges 

MAX_DOMAIN_KEY_SIZE = 200
MAX_PHONE_KEY_SIZE  = 50
MAX_EMAIL_KEY_SIZE  = 50

# Domain edges:exact domain +same country+ cap
dom = (
    df.where(F.col("domain_final").isNotNull() & (F.col("domain_final") != ""))
      .select(
          F.col("record_id").alias("id"),
          F.col("domain_final").alias("key"),
          F.col("country_code_norm").alias("cc")
      )
)

dom_sizes = dom.groupBy("key").count().withColumnRenamed("count", "key_cnt")
dom_good = dom.join(dom_sizes, "key").where(F.col("key_cnt") <= MAX_DOMAIN_KEY_SIZE).select("id", "key", "cc")

domain_edges = (
    dom_good.select("key", "cc", F.col("id").alias("id1"))
            .join(dom_good.select("key", "cc", F.col("id").alias("id2")), on=["key", "cc"])
            .where(F.col("id1") < F.col("id2"))
            .select(
                "id1", "id2",
                F.lit("det_domain_cc").alias("match_type"),
                F.lit(100.0).alias("score"),
                F.concat(F.lit("same domain+country: "), F.col("key"), F.lit("|"), F.col("cc")).alias("reason")
            )
)

# Phone edges: exact phone +cap
phones = (
    df.select(
        "record_id",
        F.col("country_code_norm").alias("cc"),
        F.explode("phone_numbers_norm_arr").alias("phone")
    )
    .where(F.col("phone").isNotNull() & (F.col("phone") != ""))
)

phone_sizes = phones.groupBy("phone").count().withColumnRenamed("count", "key_cnt")
phones_good = phones.join(phone_sizes, "phone").where(F.col("key_cnt") <= MAX_PHONE_KEY_SIZE).select("record_id", "phone", "cc")

phone_edges = (
    phones_good.select(F.col("phone").alias("key"), "cc", F.col("record_id").alias("id1"))
              .join(phones_good.select(F.col("phone").alias("key"), "cc", F.col("record_id").alias("id2")), on=["key", "cc"])
              .where(F.col("id1") < F.col("id2"))
              .select(
                  "id1", "id2",
                  F.lit("det_phone_cc").alias("match_type"),
                  F.lit(100.0).alias("score"),
                  F.concat(F.lit("same phone+country: "), F.col("key"), F.lit("|"), F.col("cc")).alias("reason")
              )
)

# Email edges: exact email +cap
emails = (
    df.where(F.col("primary_email_norm").isNotNull() & (F.col("primary_email_norm") != ""))
      .select(
          F.col("record_id").alias("id"),
          F.col("primary_email_norm").alias("key"),
          F.col("country_code_norm").alias("cc")
      )
)

email_sizes = emails.groupBy("key").count().withColumnRenamed("count", "key_cnt")
emails_good = emails.join(email_sizes, "key").where(F.col("key_cnt") <= MAX_EMAIL_KEY_SIZE).select("id", "key", "cc")

email_edges = (
    emails_good.select("key", "cc", F.col("id").alias("id1"))
              .join(emails_good.select("key", "cc", F.col("id").alias("id2")), on=["key", "cc"])
              .where(F.col("id1") < F.col("id2"))
              .select(
                  "id1", "id2",
                  F.lit("det_email_cc").alias("match_type"),
                  F.lit(100.0).alias("score"),
                  F.concat(F.lit("same email+country: "), F.col("key"), F.lit("|"), F.col("cc")).alias("reason")
              )
)

# Candidate generation via blocking
blocks = df.select("record_id", F.explode("blocking_keys").alias("block_key"))

MAX_BLOCK_SIZE = 500
block_sizes = blocks.groupBy("block_key").count()
good_blocks = block_sizes.where(F.col("count") <= MAX_BLOCK_SIZE)
blocks_small = blocks.join(good_blocks.select("block_key"), on="block_key", how="inner")

pairs = (
    blocks_small.alias("a")
    .join(blocks_small.alias("b"), on="block_key")
    .where(F.col("a.record_id") < F.col("b.record_id"))
    .select(
        F.col("a.record_id").alias("id1"),
        F.col("b.record_id").alias("id2"),
        F.col("block_key")
    )
    .dropDuplicates(["id1", "id2"])
)

# Enriching pairs with fields for fuzzy scoring
left = df.select(
    F.col("record_id").alias("id1"),
    F.col("best_name_norm").alias("name1"),
    F.col("domain_final").alias("dom1"),
    F.col("phone_any_norm").alias("ph1"),
    F.col("primary_email_norm").alias("em1"),
    F.col("email_domain").alias("emd1"),
    F.col("country_code_norm").alias("cc1"),
    F.col("main_city_norm").alias("city1"),
    F.col("main_postcode_norm").alias("pc1"),
    F.col("lat_r3").alias("lat1"),
    F.col("lon_r3").alias("lon1"),
)

right = df.select(
    F.col("record_id").alias("id2"),
    F.col("best_name_norm").alias("name2"),
    F.col("domain_final").alias("dom2"),
    F.col("phone_any_norm").alias("ph2"),
    F.col("primary_email_norm").alias("em2"),
    F.col("email_domain").alias("emd2"),
    F.col("country_code_norm").alias("cc2"),
    F.col("main_city_norm").alias("city2"),
    F.col("main_postcode_norm").alias("pc2"),
    F.col("lat_r3").alias("lat2"),
    F.col("lon_r3").alias("lon2"),
)

pairs_enriched = pairs.join(left, "id1").join(right, "id2")

# Fuzzy name scoring (RapidFuzz UDF)
from pyspark.sql.types import DoubleType
from pyspark.sql.functions import udf
from rapidfuzz.fuzz import token_set_ratio

@udf(DoubleType())
def name_sim(a, b):
    if a is None or b is None:
        return 0.0
    return float(token_set_ratio(a, b))

pairs_scored = pairs_enriched.withColumn("name_score", name_sim(F.col("name1"), F.col("name2")))

pairs_fuzzy = pairs_scored.withColumn(
    "is_match",
    (F.col("name_score") >= 90) &
    (
        # strong corroboration
        (F.col("pc1").isNotNull() & (F.col("pc1") == F.col("pc2"))) |
        (
            F.col("cc1").isNotNull() & (F.col("cc1") == F.col("cc2")) &
            F.col("city1").isNotNull() & (F.col("city1") == F.col("city2"))
        ) |
        (
            F.col("lat1").isNotNull() & (F.col("lat1") == F.col("lat2")) &
            F.col("lon1").isNotNull() & (F.col("lon1") == F.col("lon2"))
        ) |
        # weaker corroboration (email domain) only if ALSO same country
        (
            F.col("emd1").isNotNull() & (F.col("emd1") == F.col("emd2")) &
            F.col("cc1").isNotNull() & (F.col("cc1") == F.col("cc2"))
        )
    )
)

fuzzy_edges = (
    pairs_fuzzy.where(F.col("is_match"))
    .select(
        "id1", "id2",
        F.lit("fuzzy_name_plus").alias("match_type"),
        F.col("name_score").alias("score"),
        F.concat(F.lit("name_score="), F.col("name_score").cast("string"),
                 F.lit(", block="), F.col("block_key")).alias("reason")
    )
)

# Union of all edges
all_edges = (
    domain_edges
    .unionByName(phone_edges)
    .unionByName(email_edges)
    .unionByName(fuzzy_edges)
).cache()

print("all_edges =", all_edges.count())
all_edges.show(50, truncate=False)

all_edges.groupBy("match_type").count().orderBy(F.desc("count")).show(truncate=False)


all_edges = 162872
+-----------+-----------+-------------+-----+----------------------------------------------+
|id1        |id2        |match_type   |score|reason                                        |
+-----------+-----------+-------------+-----+----------------------------------------------+
|17179869184|17179894537|det_domain_cc|100.0|same domain+country: owensliquors.com|US      |
|17179869184|17179891635|det_domain_cc|100.0|same domain+country: owensliquors.com|US      |
|17179869184|17179881325|det_domain_cc|100.0|same domain+country: owensliquors.com|US      |
|17179869184|17179879605|det_domain_cc|100.0|same domain+country: owensliquors.com|US      |
|17179869184|17179873919|det_domain_cc|100.0|same domain+country: owensliquors.com|US      |
|17179869185|17179895774|det_domain_cc|100.0|same domain+country: clubtarneit.com.au|AU    |
|17179869185|17179891911|det_domain_cc|100.0|same domain+country: clubtarneit.com.au|AU    |
|17179869185|17179884267|det_domain_cc|100.0|same d

In [12]:
# -------------------------
# 6) Connected components (label propagation fallback)
# -------------------------
use_graphframes = False  # keep fallback mode

edges_raw = (
    all_edges.select(
        F.col("id1").cast("long").alias("src"),
        F.col("id2").cast("long").alias("dst"),
        F.col("match_type"),
        F.col("score"),
        F.col("reason")
    )
    .where(F.col("src").isNotNull() & F.col("dst").isNotNull())
    .dropDuplicates(["src", "dst"])
)

# Cut lineage by writing minimal edges and re-reading
edges_base_plan = edges_raw.select("src", "dst").dropDuplicates(["src", "dst"])
tmp_edges_path = "/tmp/er_edges_base_parquet"
edges_base_plan.write.mode("overwrite").parquet(tmp_edges_path)

edges_base = (
    spark.read.parquet(tmp_edges_path)
    .select(F.col("src").cast("long"), F.col("dst").cast("long"))
    .dropDuplicates(["src", "dst"])
    .persist(StorageLevel.MEMORY_AND_DISK)
)
_ = edges_base.count()

edges_rev = edges_base.selectExpr("dst as src", "src as dst")

edges = (
    edges_base.union(edges_rev)
    .dropDuplicates(["src", "dst"])
    .persist(StorageLevel.MEMORY_AND_DISK)
)
_ = edges.count()
edges = edges.checkpoint(eager=True)

vertices = (
    df.select(F.col("record_id").cast("long").alias("id"))
    .dropDuplicates(["id"])
    .persist(StorageLevel.MEMORY_AND_DISK)
)
_ = vertices.count()

print("vertices =", vertices.count())
print("edges =", edges.count())

if not use_graphframes:
    labels = (
        vertices.withColumn("label", F.col("id"))
        .repartition(200)
        .persist(StorageLevel.MEMORY_AND_DISK)
    )
    _ = labels.count()

    edges_lp = (
        edges.select("src", "dst")
        .repartition(200)
        .persist(StorageLevel.MEMORY_AND_DISK)
    )
    _ = edges_lp.count()

    MAX_ITERS = 30
    for i in range(MAX_ITERS):
        msg_to_dst = (
            edges_lp.join(labels.select(F.col("id").alias("src"), "label"), on="src", how="inner")
            .select(F.col("dst").alias("id"), F.col("label").alias("candidate_label"))
        )

        candidates = labels.select("id", F.col("label").alias("candidate_label")).union(msg_to_dst)

        labels_next = (
            candidates.groupBy("id")
            .agg(F.min("candidate_label").alias("label"))
            .repartition(200)
            .persist(StorageLevel.MEMORY_AND_DISK)
        )

        labels_next = labels_next.checkpoint(eager=True)
        _ = labels_next.count()

        labels.unpersist()
        labels = labels_next

        print(f"Iteration {i+1} complete")

    entity_map = labels.select(
        F.col("id").alias("record_id"),
        F.col("label").cast("string").alias("entity_id")
    )

print("entity_map rows =", entity_map.count())
entity_map.show(20, truncate=False)

# -------------------------
# 6.3 Sanity checks + over-merge detectors
# -------------------------
check_all = (
    df.select("record_id").dropDuplicates()
    .join(entity_map, on="record_id", how="left")
)
missing = check_all.where(F.col("entity_id").isNull()).count()
total = check_all.count()
print("total records:", total)
print("missing entity_id:", missing)

entity_map = (
    check_all.withColumn(
        "entity_id",
        F.when(F.col("entity_id").isNotNull(), F.col("entity_id"))
        .otherwise(F.col("record_id").cast("string"))
    )
    .select("record_id", "entity_id")
)

num_entities = entity_map.select("entity_id").dropDuplicates().count()
print("unique entities:", num_entities)

cluster_sizes = (
    entity_map.groupBy("entity_id")
    .agg(F.count("*").alias("cluster_size"))
    .orderBy(F.desc("cluster_size"))
)
cluster_sizes.show(30, truncate=False)

# ---- Over-merge detector: name diversity inside clusters ----
name_diversity = (
    entity_map.join(df.select("record_id", "name_fp"), "record_id", "left")
    .groupBy("entity_id")
    .agg(
        F.count("*").alias("n"),
        F.countDistinct("name_fp").alias("distinct_name_fp")
    )
    .withColumn("name_div_ratio", F.col("distinct_name_fp") / F.col("n"))
    .orderBy(F.desc("n"))
)

print("Suspicious clusters (big + diverse names):")
name_diversity.where((F.col("n") >= 20) & (F.col("name_div_ratio") > 0.4)).show(50, truncate=False)

# Inspect a few biggest clusters
inspect_cols = [
    "record_id",
    "company_name",
    "domain_final",
    "phone_any_norm",
    "primary_email_norm",
    "country_code_norm",
    "main_city_norm",
    "main_postcode_norm"
]
df_small = df.select(*inspect_cols)

top_entities = [r["entity_id"] for r in cluster_sizes.limit(5).select("entity_id").collect()]
for eid in top_entities:
    print("\n==============================")
    print("ENTITY:", eid)
    print("==============================")
    (
        entity_map.where(F.col("entity_id") == eid)
        .join(df_small, on="record_id", how="left")
        .show(50, truncate=False)
    )

# -------------------------
# 6.4 Evidence table (audit)
# -------------------------
edges_with_entity = (
    edges_raw
    .join(entity_map.select(F.col("record_id").alias("src"), "entity_id"), on="src", how="left")
    .join(entity_map.select(F.col("record_id").alias("dst"), F.col("entity_id").alias("entity_id_dst")), on="dst", how="left")
    .where(F.col("entity_id") == F.col("entity_id_dst"))
    .drop("entity_id_dst")
)

edges_with_entity.groupBy("match_type").count().orderBy(F.desc("count")).show(truncate=False)


vertices = 33446
edges = 201338
Iteration 1 complete
Iteration 2 complete
Iteration 3 complete
Iteration 4 complete
Iteration 5 complete
Iteration 6 complete
Iteration 7 complete
Iteration 8 complete
Iteration 9 complete
Iteration 10 complete
Iteration 11 complete
Iteration 12 complete
Iteration 13 complete
Iteration 14 complete
Iteration 15 complete
Iteration 16 complete
Iteration 17 complete
Iteration 18 complete
Iteration 19 complete
Iteration 20 complete
Iteration 21 complete
Iteration 22 complete
Iteration 23 complete
Iteration 24 complete
Iteration 25 complete
Iteration 26 complete
Iteration 27 complete
Iteration 28 complete
Iteration 29 complete
Iteration 30 complete
entity_map rows = 33446
+-----------+-----------+
|record_id  |entity_id  |
+-----------+-----------+
|17179900221|17179875167|
|17179885110|17179874259|
|17179873577|17179871592|
|17179879546|17179878492|
|17179901809|17179869683|
|17179879257|17179879181|
|17179872472|17179870325|
|17179901052|17179869639|
|171798

In [13]:
# Outputs (updated dataset+ map + evidence)

final_map = entity_map.select("record_id", "entity_id")

# record_id ->entity_id mapping
out_map_path = "entity_map_parquet"
final_map.write.mode("overwrite").parquet(out_map_path)
print("Wrote final_map to:", out_map_path)

# UPDATED DATASET: original columns (+ record_id) + entity_id
updated_dataset_path = "veridion_with_entity_id_parquet"
updated_df = df_raw.join(final_map, on="record_id", how="left")

missing_updated = updated_df.where(F.col("entity_id").isNull()).count()
print("missing entity_id in updated_df:", missing_updated)

updated_df.write.mode("overwrite").parquet(updated_dataset_path)
print("Wrote updated dataset to:", updated_dataset_path)

# Evidence table
evidence_path = "entity_edges_evidence_parquet"
evidence_df = edges_with_entity.select("entity_id", "src", "dst", "match_type", "score", "reason")
evidence_df.write.mode("overwrite").parquet(evidence_path)
print("Wrote evidence table to:", evidence_path)


Wrote final_map to: entity_map_parquet
missing entity_id in updated_df: 0
Wrote updated dataset to: veridion_with_entity_id_parquet
Wrote evidence table to: entity_edges_evidence_parquet
